# Tutorial 5: Templated Generation

This notebook shows how to generate new text on a different topic while preserving the style of a source text using `TemplatedGenerator`.

Use this when you have an example text with a style you like, but want new content about a caller-provided target topic.

By the end of this tutorial you will know how to:
1. Generate one templated text sample
2. Generate multiple templated samples as a DataFrame
3. Inspect source-topic and style metadata
4. Run the generator with a mock provider or OpenAI
5. Save generated samples for later use

## Installation

Install the library with the OpenAI provider if you want to call a live model:

In [ ]:
# Uncomment to install from PyPI
# !pip install synthetictext[openai]

## Source Text, Topic, and Style

`TemplatedGenerator` needs four core inputs:
- `text`: the source text whose style should be preserved
- `source_topic`: what the source text is about
- `target_topic`: what the generated text should be about
- `style`: a short style description to preserve

In [ ]:
source_text = """
Battery costs are falling, charging networks are expanding, and electric vehicle adoption is moving from early adopters into the mainstream. The next phase will depend less on novelty and more on reliability, affordability, and the everyday confidence of drivers.
""".strip()

source_topic = "electric cars"
target_topic = "urban gardening"
style = "brief analytical market update"

print(source_text)

## Run Without an API Key

For tutorials, tests, and local experimentation, you can pass any object that implements the `BaseLLMProvider.generate(...)` interface. This mock provider returns deterministic JSON so the notebook runs without a live API call.

In [ ]:
from synthetictext import BaseLLMProvider, TemplatedGenerator


class MockTemplatedLLMProvider(BaseLLMProvider):
    def generate(
        self,
        prompt: str,
        *,
        model=None,
        temperature=0.9,
        max_tokens=250,
        system_prompt=None,
        response_format=None,
    ) -> str:
        # TemplatedGenerator asks OpenAI-compatible providers for JSON mode.
        assert response_format == {"type": "json_object"}
        return '''{
            "samples": [
                {
                    "text": "Soil costs are manageable, community plots are expanding, and urban gardening is moving from hobbyists into everyday city life. The next phase will depend less on novelty and more on access, consistency, and the everyday confidence of new growers."
                },
                {
                    "text": "Balcony planters are improving, local seed networks are expanding, and urban gardening is shifting from niche interest to practical household habit. The next phase will depend less on enthusiasm and more on space, guidance, and reliable growing routines."
                }
            ]
        }'''


## Generate One Text

`generate_one(...)` is a convenience method when you only need a single string.

In [ ]:
generator = TemplatedGenerator(llm_provider=MockTemplatedLLMProvider())

generated_text = generator.generate_one(
    text=source_text,
    source_topic=source_topic,
    target_topic=target_topic,
    style=style,
    language="English",
)

print(generated_text)

## Generate Multiple Samples

`generate(...)` returns a pandas DataFrame with one row per generated sample.

In [ ]:
templated_df = generator.generate(
    text=source_text,
    source_topic=source_topic,
    target_topic=target_topic,
    style=style,
    num_samples=2,
    language="English",
)

templated_df

## Inspect Metadata

Each row includes metadata that records the style-transfer setup and a hash of the source text.

In [ ]:
templated_df.loc[0, "metadata"]

## Use OpenAI

For real generation, use the OpenAI provider shorthand. `TemplatedGenerator` requests OpenAI JSON mode so the model is constrained to return a JSON object.

In [ ]:
import os

# Uncomment after setting OPENAI_API_KEY in your environment.
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this cell"
openai_generator = TemplatedGenerator(llm_provider="openai", llm_model="gpt-5.5")
openai_df = openai_generator.generate(
    text=source_text,
    source_topic=source_topic,
    target_topic=target_topic,
    style=style,
    num_samples=5,
    language="Telugu",
)
openai_df

## Save Samples

The returned DataFrame can be saved directly to CSV.

In [ ]:
output_path = "templated_samples.csv"
templated_df.to_csv(output_path, index=False)
print(f"Saved {len(templated_df)} samples to {output_path}")

## CLI Equivalent

You can also generate templated samples from a source text file using the CLI:

```bash
synthetictext templated \
  --input source.txt \
  --source-topic "electric vehicles" \
  --target-topic "urban gardening" \
  --style "brief analytical market update" \
  --num-samples 5 \
  --output templated.csv
```